# Wattson on a real NVIDIA GPU

Measures **joules per token** for LLM inference on a Kaggle GPU, using the
Wattson gateway (Rust) with real NVML energy readings.

**Before running:** in the right-hand panel, set **Accelerator → GPU T4 x2**
and turn **Internet → On**. Then run the cells top to bottom.

```
load_test.py  ->  wattson gateway (:8080, NVML)  ->  model server (:8000, GPU 0)
```

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=index,name,power.draw,power.limit,memory.total --format=csv

## 2. Install Rust and build Wattson (about 2 minutes)

In [ ]:
%%bash
curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal > /dev/null 2>&1
cd /kaggle/working
rm -rf wattson && git clone -q https://github.com/Shub1806/wattson.git
cd wattson && $HOME/.cargo/bin/cargo build --release 2>&1 | tail -2

## 3. Start the model server and the gateway

The model server is the simple baseline: Hugging Face `transformers`, one
request at a time on GPU 0. The gateway reads GPU 0's energy through NVML.

In [ ]:
import os, subprocess, time, urllib.request

os.chdir("/kaggle/working/wattson")
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

def wait_for(url, proc, log, timeout=600):
    start = time.time()
    while time.time() - start < timeout:
        if proc.poll() is not None:
            print(open(log).read()[-3000:])
            raise RuntimeError(f"process exited, see {log}")
        try:
            urllib.request.urlopen(url, timeout=2)
            return
        except Exception:
            time.sleep(2)
    raise TimeoutError(url)

# Same GPU numbering for CUDA and NVML, and only GPU 0 for the model.
base_env = dict(os.environ, CUDA_DEVICE_ORDER="PCI_BUS_ID")

model_server = subprocess.Popen(
    ["python", "-u", "scripts/hf_server.py"],
    env=dict(base_env, CUDA_VISIBLE_DEVICES="0", MODEL=MODEL, PORT="8000"),
    stdout=open("hf_server.log", "w"), stderr=subprocess.STDOUT)
wait_for("http://127.0.0.1:8000/health", model_server, "hf_server.log")
print("model server ready")

gateway = subprocess.Popen(
    ["./target/release/wattson"],
    env=dict(base_env, WATTSON_ENERGY="nvml", WATTSON_GPU_INDEX="0"),
    stdout=open("wattson.log", "w"), stderr=subprocess.STDOUT)
wait_for("http://127.0.0.1:8080/health", gateway, "wattson.log", timeout=30)
print(open("wattson.log").read())

## 4. Smoke test: one request through the gateway

In [ ]:
!curl -sN localhost:8080/v1/chat/completions -H 'Content-Type: application/json' \
  -d '{"stream":true,"max_tokens":64,"messages":[{"role":"user","content":"What is a GPU?"}]}' | tail -3
!echo; tail -1 wattson.log; curl -s localhost:8080/energy

## 5. Benchmark

Sends 1, 2 and 4 simultaneous requests (3 rounds each, 128 tokens per
request) and saves `results/<label>.csv`. If you got a P100 instead of a T4,
change the label to `hf-serial-p100`.

In [ ]:
!python scripts/load_test.py --label hf-serial-t4 --concurrency 1,2,4 --waves 3 --max-tokens 128

## 6. Gateway log and energy totals

In [ ]:
!tail -15 wattson.log; echo; curl -s localhost:8080/energy

## 7. Stop everything

Then download `results/hf-serial-t4.csv` from the **Output** panel
(`/kaggle/working/wattson/results/`), put it in your repo's `results/`
folder, and commit it.

In [ ]:
gateway.terminate(); model_server.terminate()
print("stopped")